## Imports  

Update 2024 on of the 2024 WP vintage with the difference between the 2024 WP vintage's 2023 value and the 2023 WP vintage's 2023 value

In [100]:
import pandas as pd
import sqlite3 as sq
import matplotlib as mpl
from matplotlib import rcParams
import matplotlib.pyplot as plt
import numpy as np
pd.set_option('display.max_rows', 1000); pd.set_option('display.max_columns', 1000); pd.set_option('display.width', 1000)
pd.options.mode.chained_assignment = None  # default='warn'

## Functions

In [101]:
def mantissa_round(x):
    y = np.floor(x)
    indices = np.argsort(x-y)
    hilowman = indices[::-1]
    diffxy = np.sum(x)-np.sum(y)
    z=0
    while z < diffxy:
        #z single dimemsion - need to adjust to take table
        y[hilowman[z]] +=1
        z +=1
    return y

## Dictionaries

In [102]:
wptobeadroppercents = {'WP%:Farm': 'Farm employment', 'WP%:Forestry, Fishing, & Related': 'Forestry, fishing, and related activities', 
                       'WP%:Mining': 'Mining, quarrying, and oil and gas extraction', 'WP%:Construction': 'Construction', 'WP%:Manufacturing': 'Manufacturing', 
                       'WP%:Utilities': 'Utilities', 'WP%:Wholesale Trade': 'Wholesale trade', 'WP%:Retail Trade': 'Retail trade', 
                       'WP%:Transportation & Warehousing': 'Transportation and warehousing', 'WP%:Information': 'Information', 
                       'WP%:Finance & Insurance': 'Finance and insurance', 'WP%:Real Estate, Rental, & Leasing': 'Real estate and rental and leasing', 
                       'WP%:Professional & Technical Services': 'Professional, scientific, and technical services', 
                       'WP%:Management of Companies & Enterprises': 'Management of companies and enterprises',
                       'WP%:Administrative & Waste Services': 'Administrative and support and waste management and remediation services', 
                       'WP%:Educational Services': 'Educational services', 'WP%:Healthcare & Social Assistance': 'Health care and social assistance', 
                       'WP%:Arts, Entertainment, & Recreation': 'Arts, entertainment, and recreation',
                       'WP%:Accommodation & Food Services': 'Accommodation and food services', 
                       'WP%:Other': 'Other services (except government and government enterprises)', 'WP%:Federal Civilian': 'Federal civilian', 
                       'WP%:Federal Military': 'Military', 'WP%:State & Local Government': 'State and local'}

In [103]:
wptobea = {'WP:Total': 'Total employment (number of jobs)', 'WP:Farm': 'Farm employment', 
           'WP:Forestry, Fishing, & Related': 'Forestry, fishing, and related activities', 'WP:Mining': 'Mining, quarrying, and oil and gas extraction', 
           'WP:Construction': 'Construction', 'WP:Manufacturing': 'Manufacturing', 'WP:Utilities': 'Utilities','WP:Wholesale Trade': 'Wholesale trade', 
           'WP:Retail Trade': 'Retail trade', 'WP:Transportation & Warehousing': 'Transportation and warehousing', 'WP:Information': 'Information', 
           'WP:Finance & Insurance': 'Finance and insurance','WP:Real Estate, Rental, & Leasing': 'Real estate and rental and leasing', 
           'WP:Professional & Technical Services': 'Professional, scientific, and technical services', 
           'WP:Management of Companies & Enterprises': 'Management of companies and enterprises',
           'WP:Administrative & Waste Services': 'Administrative and support and waste management and remediation services', 
           'WP:Educational Services': 'Educational services', 
           'WP:Healthcare & Social Assistance': 'Health care and social assistance', 'WP:Arts, Entertainment, & Recreation': 'Arts, entertainment, and recreation',
           'WP:Accommodation & Food Services': 'Accommodation and food services', 'WP:Other': 'Other services (except government and government enterprises)', 
           'WP:Federal Civilian': 'Federal civilian', 'WP:Federal Military': 'Military', 'WP:State & Local Government': 'State and local', 
           
           'WP%:Farm': 'Farm employment %', 'WP%:Forestry, Fishing, & Related': 'Forestry, fishing, and related activities %', 
           'WP%:Mining': 'Mining, quarrying, and oil and gas extraction %', 'WP%:Construction': 'Construction %', 'WP%:Manufacturing': 'Manufacturing %', 
           'WP%:Utilities': 'Utilities %', 'WP%:Wholesale Trade': 'Wholesale trade %', 'WP%:Retail Trade': 'Retail trade %', 
           'WP%:Transportation & Warehousing': 'Transportation and warehousing %', 'WP%:Information': 'Information %', 
           'WP%:Finance & Insurance': 'Finance and insurance %', 'WP%:Real Estate, Rental, & Leasing': 'Real estate and rental and leasing %', 
           'WP%:Professional & Technical Services': 'Professional, scientific, and technical services %', 
           'WP%:Management of Companies & Enterprises': 'Management of companies and enterprises %',
           'WP%:Administrative & Waste Services': 'Administrative and support and waste management and remediation services %', 
           'WP%:Educational Services': 'Educational services %', 'WP%:Healthcare & Social Assistance': 'Health care and social assistance %', 
           'WP%:Arts, Entertainment, & Recreation': 'Arts, entertainment, and recreation %', 'WP%:Accommodation & Food Services': 'Accommodation and food services %', 
           'WP%:Other': 'Other services (except government and government enterprises) %', 'WP%:Federal Civilian': 'Federal civilian %', 
           'WP%:Federal Military': 'Military %', 'WP%:State & Local Government': 'State and local %'}

# Initial 2023 Values

In [104]:
#this is based on 2022 value imputed by me
wp23 = pd.read_csv('../data/2023Imputed_2023WPVintage_2022BEA.csv')
#this is actually based on 2022 value imputed by woods and poole
wp24 = pd.read_csv('../data/2023Imputed_2024WPVintage_2022BEA.csv')
#so really the shares of the leftover employment (difference between total employment and non-suppressed industries) are slightly different

In [105]:
wp23.head()

,Industry,NAME,2023
0,Accommodation and food services,"Cheatham County, Tennessee",911.678571
1,Accommodation and food services,"Davidson County, Tennessee",63675.392857
2,Accommodation and food services,"Dickson County, Tennessee",2245.035714
3,Accommodation and food services,"Houston County, Tennessee",147.928571
4,Accommodation and food services,"Humphreys County, Tennessee",724.785714


In [106]:
df = wp23.pivot(index = 'NAME', columns = 'Industry', values = '2023').reset_index(drop = False)
df['Year'] = '2023_2023WP'
df.head(2)

Industry,NAME,Accommodation and food services,Administrative and support and waste management and remediation services,"Agriculture, forestry, fishing and hunting","Arts, entertainment, and recreation",Construction,Educational services,Farm employment,Finance and insurance,"Forestry, fishing, and related activities",Government,Health care and social assistance,Information,Management of companies and enterprises,Manufacturing,"Mining, quarrying, and oil and gas extraction",Other services (except government and government enterprises),"Professional, scientific, and technical services",Real estate and rental and leasing,Retail trade,Total employment (number of jobs),Transportation and warehousing,Utilities,Wholesale trade,Year
0,"Cheatham County, Tennessee",911.678571,1011.892857,486.515151,584.250000,1948.571429,272.033305,427.678571,590.928571,58.836579,1786.321429,827.980806,216.464286,126.941857,2928.464286,17.433060,1153.214286,783.025119,913.964286,1492.500000,17292.0,970.250000,38.224386,231.346316,2023_2023WP
1,"Davidson County, Tennessee",63675.392857,64716.607143,682.642857,32852.535714,41425.821429,27978.821429,451.500000,44982.607143,231.142857,48155.357143,95574.857143,21871.821429,16630.535714,22708.928571,706.821429,38789.678571,67291.107143,39858.607143,51251.857143,753485.5,47951.964286,391.785714,25987.750000,2023_2023WP


In [107]:
df1 = wp24.pivot(index = 'NAME', columns = 'Industry', values = '2023').reset_index(drop = False)
df1['Year'] = '2023_2024WP'
df1.head(2)

Industry,NAME,Accommodation and food services,Administrative and support and waste management and remediation services,"Agriculture, forestry, fishing and hunting","Arts, entertainment, and recreation",Construction,Educational services,Farm employment,Finance and insurance,"Forestry, fishing, and related activities",Government,Health care and social assistance,Information,Management of companies and enterprises,Manufacturing,"Mining, quarrying, and oil and gas extraction",Other services (except government and government enterprises),"Professional, scientific, and technical services",Real estate and rental and leasing,Retail trade,Total employment (number of jobs),Transportation and warehousing,Utilities,Wholesale trade,Year
0,"Cheatham County, Tennessee",910.0,1033.0,483.022027,593.0,2037.0,273.110542,428.0,619.0,55.022027,1787.0,788.316942,226.0,101.04079,2956.0,18.007342,1157.0,829.336521,949.0,1531.0,17628.964286,1018.0,42.016724,278.113397,2023_2024WP
1,"Davidson County, Tennessee",62294.0,63697.0,691.000000,32735.0,42651.0,27012.000000,456.0,44782.0,235.000000,48076.0,99314.000000,21880.0,16788.00000,23127.0,718.000000,38323.0,67064.000000,40823.0,51774.0,754716.000000,46752.0,399.000000,25816.000000,2023_2024WP


# Smoothing

In [108]:
conn = sq.connect('../../Data-Pipelines/Outputs/WoodsandPooleandAffiliated.db')
sql_query = pd.read_sql('SELECT * FROM [WP2024_IndustryEmployment_Annual_Change_UnadjustedNoReplacements]', conn)
emp = pd.DataFrame(sql_query)
emp = emp.loc[emp['Year'] != 'None']
emp['Year'] = emp['Year'].astype(int)
emp= emp[emp['Year'] > 2009]
# Create a boolean array indicating which columns contain the string "%"
cols_to_drop = emp.columns[emp.columns.str.contains('%')]
# Drop the columns containing the string "%"
emp.drop(cols_to_drop, axis=1, inplace=True)
cols_to_drop = emp.columns[emp.columns.str.contains('Change')]
# Drop the columns containing the string "Change"
emp.drop(cols_to_drop, axis=1, inplace=True)
#filter and renname
emp['Farm employment'] = emp['WP:Farm']
emp['Forestry, fishing, and related activities'] = emp['WP:Forestry, Fishing, & Related']
emp['Agriculture, forestry, fishing and hunting'] = emp['WP:Farm'] + emp['WP:Forestry, Fishing, & Related']
thelist = [emp['WP:Federal Civilian'], emp['WP:Federal Military'], emp['WP:State & Local Government']]
emp['Government'] = sum(thelist)
emp = emp.drop(columns = ['WP:Farm', 'WP:Federal Civilian', 'WP:Federal Military','WP:Forestry, Fishing, & Related', 'WP:State & Local Government', 
                         'Ind:Construction', 'Ind:Education & Health Services', 'Ind:Farm', 'Ind:Financial Activities','Ind:Goods Producing', 'Ind:Information', 
                          'Ind:Leisure & Hospitality', 'Ind:Manufacturing', 'Ind:Natural Resources & Mining', 'Ind:Other','Ind:Professional & Business Services',
                          'Ind:Public Administration', 'Ind:Service Producing', 'Ind:Snapshot: Blue Collar','Ind:Snapshot: Farm','Ind:Snapshot: Government', 
                          'Ind:Snapshot: Industrial, Manufacturing, Utilities, & Logistics','Ind:Snapshot: Professional Services', 'Time Frame', 
                          'Ind:Snapshot: Retail & Hospitality', 'Ind:Snapshot: White Collar', 'Ind:Total', 'Source', 'GEO_ID', 'Ind:Trade, Transportation, & Utilities'])
inddict = {'WP:Accommodation & Food Services': 'Accommodation and food services', 
           'WP:Administrative & Waste Services': 'Administrative and support and waste management and remediation services',
           'WP:Arts, Entertainment, & Recreation': 'Arts, entertainment, and recreation', 
           'WP:Construction': 'Construction', 
           'WP:Educational Services': 'Educational services', 
           'WP:Finance & Insurance': 'Finance and insurance', 
           'WP:Healthcare & Social Assistance': 'Health care and social assistance', 
           'WP:Information': 'Information',
           'WP:Management of Companies & Enterprises': 'Management of companies and enterprises', 
           'WP:Manufacturing': 'Manufacturing', 
           'WP:Mining': 'Mining, quarrying, and oil and gas extraction', 
           'WP:Other': 'Other services (except government and government enterprises)', 
           'WP:Professional & Technical Services': 'Professional, scientific, and technical services',
           'WP:Real Estate, Rental, & Leasing': 'Real estate and rental and leasing', 
           'WP:Retail Trade': 'Retail trade', 
           'WP:Total': 'Total employment (number of jobs)', 
           'WP:Transportation & Warehousing': 'Transportation and warehousing',
           'WP:Utilities': 'Utilities', 
           'WP:Wholesale Trade': 'Wholesale trade'}
emp = emp.rename(columns = inddict)
emp['Year'] = emp['Year'].astype(str)
#emp['Year'] = emp['Year'].replace({'2022':'2022WP'})
emp = emp.loc[emp['Year'] != '2023']
#select only counties we want, narrow in on types of employment, and select only 2022 (most recent year)
thelist = ['Cheatham County, Tennessee', 'Davidson County, Tennessee', 'Dickson County, Tennessee', 'Houston County, Tennessee', 
           'Humphreys County, Tennessee', 'Maury County, Tennessee', 'Montgomery County, Tennessee', 'Robertson County, Tennessee', 
           'Rutherford County, Tennessee', 'Stewart County, Tennessee', 'Sumner County, Tennessee', 'Williamson County, Tennessee', 
           'Wilson County, Tennessee', 'Trousdale County, Tennessee']
emp = emp.loc[emp['NAME'].isin(thelist)]
emp.head(2)

,NAME,Year,Accommodation and food services,Administrative and support and waste management and remediation services,"Arts, entertainment, and recreation",Construction,Educational services,Finance and insurance,Health care and social assistance,Information,Management of companies and enterprises,Manufacturing,"Mining, quarrying, and oil and gas extraction",Other services (except government and government enterprises),"Professional, scientific, and technical services",Real estate and rental and leasing,Retail trade,Total employment (number of jobs),Transportation and warehousing,Utilities,Wholesale trade,Farm employment,"Forestry, fishing, and related activities","Agriculture, forestry, fishing and hunting",Government
20,"Cheatham County, Tennessee",2010,612.0,1145.0,545.0,1461.0,169.0,358.0,634.0,113.0,126.0,2276.0,15.0,970.0,564.0,482.0,1248.0,14015.0,643.0,33.0,223.0,465.0,49.0,514.0,1884.0
21,"Cheatham County, Tennessee",2011,577.0,1022.0,454.0,1463.0,151.0,385.0,685.0,104.0,120.0,2189.0,16.0,964.0,573.0,489.0,1296.0,13764.0,713.0,31.0,238.0,446.0,53.0,499.0,1795.0


In [109]:
emp['Year'].unique()

array(['2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017',
       '2018', '2019', '2020', '2021', '2022', '2024', '2025', '2026',
       '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034',
       '2035', '2036', '2037', '2038', '2039', '2040', '2041', '2042',
       '2043', '2044', '2045', '2046', '2047', '2048', '2049', '2050'],
      dtype=object)

In [110]:
baseline = pd.concat([emp, df, df1]).reset_index(drop = True)
baseline.head(2)

,NAME,Year,Accommodation and food services,Administrative and support and waste management and remediation services,"Arts, entertainment, and recreation",Construction,Educational services,Finance and insurance,Health care and social assistance,Information,Management of companies and enterprises,Manufacturing,"Mining, quarrying, and oil and gas extraction",Other services (except government and government enterprises),"Professional, scientific, and technical services",Real estate and rental and leasing,Retail trade,Total employment (number of jobs),Transportation and warehousing,Utilities,Wholesale trade,Farm employment,"Forestry, fishing, and related activities","Agriculture, forestry, fishing and hunting",Government
0,"Cheatham County, Tennessee",2010,612.0,1145.0,545.0,1461.0,169.0,358.0,634.0,113.0,126.0,2276.0,15.0,970.0,564.0,482.0,1248.0,14015.0,643.0,33.0,223.0,465.0,49.0,514.0,1884.0
1,"Cheatham County, Tennessee",2011,577.0,1022.0,454.0,1463.0,151.0,385.0,685.0,104.0,120.0,2189.0,16.0,964.0,573.0,489.0,1296.0,13764.0,713.0,31.0,238.0,446.0,53.0,499.0,1795.0


In [111]:
baseline = baseline.melt(id_vars = ['NAME', 'Year'], var_name = 'Industry', value_name = 'Employment')
baseline = baseline.pivot(index = ['Industry', 'NAME'], columns = 'Year', values = 'Employment').reset_index()
baseline.head(2)

Year,Industry,NAME,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023_2023WP,2023_2024WP,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050
0,Accommodation and food services,"Cheatham County, Tennessee",612.0,577.0,583.0,610.0,650.0,662.0,716.0,786.0,831.0,824.0,804.0,835.0,877.0,911.678571,910.0,944.0,979.0,1015.0,1052.0,1089.0,1128.0,1167.0,1208.0,1249.0,1292.0,1336.0,1381.0,1426.0,1474.0,1522.0,1571.0,1622.0,1674.0,1728.0,1782.0,1839.0,1896.0,1955.0,2016.0,2078.0,2142.0,2207.0
1,Accommodation and food services,"Davidson County, Tennessee",41443.0,42092.0,44849.0,46526.0,49400.0,50765.0,53956.0,56158.0,59518.0,62078.0,45898.0,50924.0,59594.0,63675.392857,62294.0,63403.0,64526.0,65644.0,66748.0,67833.0,68897.0,69939.0,70958.0,71954.0,72925.0,73874.0,74797.0,75697.0,76572.0,77423.0,78249.0,79051.0,79829.0,80582.0,81310.0,82015.0,82694.0,83350.0,83982.0,84589.0,85172.0,85731.0


In [112]:
years = list(baseline.columns)
years.remove('NAME')
years.remove('Industry')
years.remove('2023_2023WP')
years.remove('2023_2024WP')
years.remove('2050')

In [113]:
base_final = baseline[['Industry', 'NAME', '2023_2023WP', '2023_2024WP', '2050']]

In [114]:
def apply_formula(row):
    # Assuming '2022' is the base year for BL and WP, and '2050' is the last year
    base_year = 2023
    final_year = 2050

    # Calculate the denominator part of the formula
    denominator = final_year - base_year
    
    # If denominator is zero, return 0 to avoid division by zero error
    if denominator == 0:
        return 0
    # Loop through the years between 2024 and 2049 (inclusive)
    for year in range(base_year + 1, final_year):
        # Calculate the adjusted value for the current year
        adjusted_value = row[f'{year}'] - (row[f'{base_year}_2023WP'] - row[f'{base_year}_2024WP']) / denominator * (final_year - int(year))
       # Update the value for the current year
        row[f'{year}'] = adjusted_value 
    return row

# Apply the formula to each row in the DataFrame
baseline = baseline.apply(apply_formula, axis=1)

So this difference is basically the difference between my imputed values and the values woods and poole imputed, by way of the 2023 value...

In [115]:
baseline.head(2)

Year,Industry,NAME,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023_2023WP,2023_2024WP,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050
0,Accommodation and food services,"Cheatham County, Tennessee",612.0,577.0,583.0,610.0,650.0,662.0,716.0,786.0,831.0,824.0,804.0,835.0,877.0,911.678571,910.0,942.383598,977.445767,1013.507937,1050.570106,1087.632275,1126.694444,1165.756614,1206.818783,1247.880952,1290.943122,1335.005291,1380.067460,1425.129630,1473.191799,1521.253968,1570.316138,1621.378307,1673.440476,1727.502646,1781.564815,1838.626984,1895.689153,1954.751323,2015.813492,2077.875661,2141.937831,2207.0
1,Accommodation and food services,"Davidson County, Tennessee",41443.0,42092.0,44849.0,46526.0,49400.0,50765.0,53956.0,56158.0,59518.0,62078.0,45898.0,50924.0,59594.0,63675.392857,62294.0,62072.769841,63246.932540,64416.095238,65571.257937,66707.420635,67822.583333,68915.746032,69985.908730,71033.071429,72055.234127,73055.396825,74029.559524,74980.722222,75906.884921,76809.047619,77686.210317,78539.373016,79368.535714,80172.698413,80951.861111,81708.023810,82438.186508,83145.349206,83828.511905,84486.674603,85120.837302,85731.0


# Government Distribution

In [116]:
dist = pd.read_csv('../data/jobseqdistr.csv')
dist.head(2)

,NAME,Industry,Share Private,Share Self-Employed,Share Government
0,"Cheatham County, Tennessee",Accommodation and food services,89.455529,2.746548,7.797922
1,"Cheatham County, Tennessee",Administrative and support and waste managemen...,52.805013,42.308646,4.886341


In [117]:
dist['Industry'].unique()

array(['Accommodation and food services',
       'Administrative and support and waste management and remediation services',
       'Agriculture, forestry, fishing and hunting',
       'Arts, entertainment, and recreation', 'Construction',
       'Educational services', 'Farm employment', 'Finance and insurance',
       'Forestry, fishing, and related activities',
       'Health care and social assistance', 'Information',
       'Management of companies and enterprises', 'Manufacturing',
       'Mining, quarrying, and oil and gas extraction',
       'Other services (except government and government enterprises)',
       'Professional, scientific, and technical services',
       'Real estate and rental and leasing', 'Retail trade',
       'Total employment (number of jobs)',
       'Transportation and warehousing', 'Utilities', 'Wholesale trade'],
      dtype=object)

In [118]:
data = baseline.merge(dist, on = ['NAME', 'Industry'], how = 'outer')
data.fillna(0, inplace = True)

In [119]:
data.head(2)

,Industry,NAME,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023_2023WP,2023_2024WP,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,Share Private,Share Self-Employed,Share Government
0,Accommodation and food services,"Cheatham County, Tennessee",612.0,577.0,583.0,610.0,650.0,662.0,716.0,786.0,831.0,824.0,804.0,835.0,877.0,911.678571,910.0,942.383598,977.445767,1013.507937,1050.570106,1087.632275,1126.694444,1165.756614,1206.818783,1247.880952,1290.943122,1335.005291,1380.067460,1425.129630,1473.191799,1521.253968,1570.316138,1621.378307,1673.440476,1727.502646,1781.564815,1838.626984,1895.689153,1954.751323,2015.813492,2077.875661,2141.937831,2207.0,89.455529,2.746548,7.797922
1,Accommodation and food services,"Davidson County, Tennessee",41443.0,42092.0,44849.0,46526.0,49400.0,50765.0,53956.0,56158.0,59518.0,62078.0,45898.0,50924.0,59594.0,63675.392857,62294.0,62072.769841,63246.932540,64416.095238,65571.257937,66707.420635,67822.583333,68915.746032,69985.908730,71033.071429,72055.234127,73055.396825,74029.559524,74980.722222,75906.884921,76809.047619,77686.210317,78539.373016,79368.535714,80172.698413,80951.861111,81708.023810,82438.186508,83145.349206,83828.511905,84486.674603,85120.837302,85731.0,98.923177,1.076823,0.000000


In [120]:
#hardcode these values in
data.loc[data['Industry'] == 'Government', 'Share Government'] = 100
#calculate numbers from the shares
cols = list(data.columns)
cols.remove('Industry')
cols.remove('NAME')
cols.remove('Share Private')
cols.remove('Share Government')
cols.remove('Share Self-Employed')
for col in cols:
    #currently don't want this value in the dataframe
    #data['{} Private'.format(col)] = data['{}'.format(col)] * (data['Share Private']/100)
    data['{} Government'.format(col)] = data['{}'.format(col)] * (data['Share Government']/100)
    #currently don't want this value in the dataframe
    #data['{} Self-Employed'.format(col)] = data['{}'.format(col)] * (data['Share Self-Employed']/100)
    data.loc[data['Industry'] == 'Government', '{} Government'.format(col)] = (data['{}'.format(col)]*data['Share Government']/100)

In [121]:
data.head()

,Industry,NAME,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023_2023WP,2023_2024WP,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,Share Private,Share Self-Employed,Share Government,2010 Government,2011 Government,2012 Government,2013 Government,2014 Government,2015 Government,2016 Government,2017 Government,2018 Government,2019 Government,2020 Government,2021 Government,2022 Government,2023_2023WP Government,2023_2024WP Government,2024 Government,2025 Government,2026 Government,2027 Government,2028 Government,2029 Government,2030 Government,2031 Government,2032 Government,2033 Government,2034 Government,2035 Government,2036 Government,2037 Government,2038 Government,2039 Government,2040 Government,2041 Government,2042 Government,2043 Government,2044 Government,2045 Government,2046 Government,2047 Government,2048 Government,2049 Government,2050 Government
0,Accommodation and food services,"Cheatham County, Tennessee",612.0,577.0,583.0,610.0,650.0,662.0,716.0,786.0,831.0,824.0,804.0,835.0,877.0,911.678571,910.0,942.383598,977.445767,1013.507937,1050.570106,1087.632275,1126.694444,1165.756614,1206.818783,1247.880952,1290.943122,1335.005291,1380.067460,1425.129630,1473.191799,1521.253968,1570.316138,1621.378307,1673.440476,1727.502646,1781.564815,1838.626984,1895.689153,1954.751323,2015.813492,2077.875661,2141.937831,2207.0,89.455529,2.746548,7.797922,47.723284,44.994011,45.461887,47.567326,50.686494,51.622245,55.833123,61.291669,64.800734,64.254879,62.695295,65.112651,68.387778,71.091986,70.961092,73.48634,76.220461,79.032561,81.92264,84.812719,87.858756,90.904794,94.10679,97.308786,100.666741,104.102674,107.616587,111.1305,114.878351,118.626201,122.452031,126.433819,130.493587,134.709313,138.925039,143.374702,147.824366,152.429988,157.191568,162.031128,167.026646,172.100143
1,Accommodation and food services,"Davidson County, Tennessee",41443.0,42092.0,44849.0,46526.0,49400.0,50765.0,53956.0,56158.0,59518.0,62078.0,45898.0,50924.0,59594.0,63675.392857,62294.0,62072.769841,63246.932540,64416.095238,65571.257937,66707.420635,67822.583333,68915.746032,69985.908730,71033.071429,72055.234127,73055.396825,74029.559524,74980.722222,75906.884921,76809.047619,77686.210317,78539.373016,79368.535714,80172.698413,80951.861111,81708.023810,82438.186508,83145.349206,83828.511905,84486.674603,85120.837302,85731.0,98.923177,1.076823,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,Accommodation and food services,"Dickson County, Tennessee",1351.0,1433.0,1365.0,1452.0,1481.0,1665.0,1811.0,1927.0,2008.0,2197.0,1962.0,1976.0,2097.0,2245.035714,2157.0,2134.224868,2200.485450,2267.746032,2336.006614,2406.267196,2477.527778,2549.788360,2624.048942,2700.309524,2777.570106,2855.830688,2936.091270,3018.351852,3101.612434,3186.873016,3274.133598,3363.394180,3453.654762,3545.915344,3640.175926,3736.436508,3834.697090,3934.957672,4037.218254,4141.478836,4248.739418,4357.0,98.636114,1.363886,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,Accommodation and food services,"Houston County, Tennessee",127.0,133.0,127.0,124.0,109.0,133.0,114.0,131.0,152.0,151.0,157.0,158.0,148.0,147.928571,154.0,164.846561,169.621693,174.396825,179.171958,183.947090,188.722222,192.497354,197.272487,202.047619,206.8

So what we need is to find the share of government employment that is distributed to which industries based on the number employed in the government in each industry where there is any government employment.

In [122]:
cols.append('NAME')
cols.append('Industry')
data = data.fillna(0)

In [123]:
data.head()

,Industry,NAME,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023_2023WP,2023_2024WP,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,Share Private,Share Self-Employed,Share Government,2010 Government,2011 Government,2012 Government,2013 Government,2014 Government,2015 Government,2016 Government,2017 Government,2018 Government,2019 Government,2020 Government,2021 Government,2022 Government,2023_2023WP Government,2023_2024WP Government,2024 Government,2025 Government,2026 Government,2027 Government,2028 Government,2029 Government,2030 Government,2031 Government,2032 Government,2033 Government,2034 Government,2035 Government,2036 Government,2037 Government,2038 Government,2039 Government,2040 Government,2041 Government,2042 Government,2043 Government,2044 Government,2045 Government,2046 Government,2047 Government,2048 Government,2049 Government,2050 Government
0,Accommodation and food services,"Cheatham County, Tennessee",612.0,577.0,583.0,610.0,650.0,662.0,716.0,786.0,831.0,824.0,804.0,835.0,877.0,911.678571,910.0,942.383598,977.445767,1013.507937,1050.570106,1087.632275,1126.694444,1165.756614,1206.818783,1247.880952,1290.943122,1335.005291,1380.067460,1425.129630,1473.191799,1521.253968,1570.316138,1621.378307,1673.440476,1727.502646,1781.564815,1838.626984,1895.689153,1954.751323,2015.813492,2077.875661,2141.937831,2207.0,89.455529,2.746548,7.797922,47.723284,44.994011,45.461887,47.567326,50.686494,51.622245,55.833123,61.291669,64.800734,64.254879,62.695295,65.112651,68.387778,71.091986,70.961092,73.48634,76.220461,79.032561,81.92264,84.812719,87.858756,90.904794,94.10679,97.308786,100.666741,104.102674,107.616587,111.1305,114.878351,118.626201,122.452031,126.433819,130.493587,134.709313,138.925039,143.374702,147.824366,152.429988,157.191568,162.031128,167.026646,172.100143
1,Accommodation and food services,"Davidson County, Tennessee",41443.0,42092.0,44849.0,46526.0,49400.0,50765.0,53956.0,56158.0,59518.0,62078.0,45898.0,50924.0,59594.0,63675.392857,62294.0,62072.769841,63246.932540,64416.095238,65571.257937,66707.420635,67822.583333,68915.746032,69985.908730,71033.071429,72055.234127,73055.396825,74029.559524,74980.722222,75906.884921,76809.047619,77686.210317,78539.373016,79368.535714,80172.698413,80951.861111,81708.023810,82438.186508,83145.349206,83828.511905,84486.674603,85120.837302,85731.0,98.923177,1.076823,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,Accommodation and food services,"Dickson County, Tennessee",1351.0,1433.0,1365.0,1452.0,1481.0,1665.0,1811.0,1927.0,2008.0,2197.0,1962.0,1976.0,2097.0,2245.035714,2157.0,2134.224868,2200.485450,2267.746032,2336.006614,2406.267196,2477.527778,2549.788360,2624.048942,2700.309524,2777.570106,2855.830688,2936.091270,3018.351852,3101.612434,3186.873016,3274.133598,3363.394180,3453.654762,3545.915344,3640.175926,3736.436508,3834.697090,3934.957672,4037.218254,4141.478836,4248.739418,4357.0,98.636114,1.363886,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,Accommodation and food services,"Houston County, Tennessee",127.0,133.0,127.0,124.0,109.0,133.0,114.0,131.0,152.0,151.0,157.0,158.0,148.0,147.928571,154.0,164.846561,169.621693,174.396825,179.171958,183.947090,188.722222,192.497354,197.272487,202.047619,206.8

In [124]:
#aside = data[['NAME', '2022 Base', 'Description', 'Government']]
#creating a couple "aside" dfs, one that has the 2022 base employment for the "government industry" 
aside1 = data.loc[data['Industry'] == 'Government']
aside1 = aside1[cols]
#and one that is the government employment per geo that isn't in the "government industry"
aside2 = data.loc[(data['Industry'] != 'Government') & (data['Industry'] != 'Total employment (number of jobs)')]
aside2 = aside2[['NAME', '2010 Government', '2011 Government', '2012 Government', '2013 Government', '2014 Government', '2015 Government', 
                 '2016 Government', '2017 Government', '2018 Government', '2019 Government', '2020 Government', '2021 Government', 
                 '2022 Government', '2023_2023WP Government', '2023_2024WP Government', '2024 Government','2025 Government','2026 Government',
                 '2027 Government','2028 Government','2029 Government','2030 Government','2031 Government','2032 Government','2033 Government',
                 '2034 Government','2035 Government','2036 Government','2037 Government','2038 Government','2039 Government','2040 Government',
                 '2041 Government','2042 Government','2043 Government','2044 Government','2045 Government','2046 Government','2047 Government',
                 '2048 Government','2049 Government','2050 Government']]

In [125]:
aside1.head(2)

,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023_2023WP,2023_2024WP,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,NAME,Industry
126,1884.0,1795.0,1730.0,1717.0,1723.0,1789.0,1774.0,1770.0,1761.0,1771.0,1728.0,1761.0,1774.0,1786.321429,1787.0,1799.653439,1811.628307,1823.603175,1833.578042,1844.552910,1854.527778,1864.502646,1873.477513,1882.452381,1891.427249,1900.402116,1908.376984,1915.351852,1922.326720,1930.301587,1937.276455,1944.251323,1950.226190,1956.201058,1962.175926,1968.150794,1974.125661,1978.100529,1983.075397,1988.050265,1993.025132,1997.0,"Cheatham County, Tennessee",Government
127,51705.0,50091.0,48260.0,47203.0,47042.0,47415.0,47906.0,47884.0,47679.0,48031.0,48647.0,46955.0,48130.0,48155.357143,48076.0,47868.582011,47727.521164,47577.460317,47420.399471,47256.338624,47087.277778,46913.216931,46737.156085,46556.095238,46371.034392,46183.973545,45993.912698,45801.851852,45605.791005,45408.730159,45208.669312,45005.608466,44801.547619,44595.486772,44387.425926,44176.365079,43964.304233,43750.243386,43535.182540,43318.121693,43098.060847,42878.0,"Davidson County, Tennessee",Government


In [126]:
aside1 = aside1.melt(id_vars = 'NAME', var_name = 'Year', value_name = 'Government Industry Employment')

In [127]:
aside1.head(2)

,NAME,Year,Government Industry Employment
0,"Cheatham County, Tennessee",2010,1884.0
1,"Davidson County, Tennessee",2010,51705.0


In [128]:
aside2 = aside2.melt(id_vars = 'NAME', var_name = 'Year', value_name = 'Employment')

In [129]:
boop = aside2['Year'].str.split(pat = " ", expand = True)
aside2['Year'] = boop[0].str.strip()

In [130]:
#group by geo to get the total non-"government industry" government employment
aside2 = aside2.groupby(['NAME', 'Year'])['Employment'].sum()
aside2 = pd.DataFrame(aside2)
aside2.reset_index(inplace = True)
aside2 = aside2.rename(columns = {'Employment': 'Non-Public Administration Government'})

In [131]:
aside2.head(2)

,NAME,Year,Non-Public Administration Government
0,"Cheatham County, Tennessee",2010,401.788060
1,"Cheatham County, Tennessee",2011,382.626479


In [132]:
aside3 = aside1.merge(aside2, on = ['NAME', 'Year'])

In [133]:
aside3.head(2)

,NAME,Year,Government Industry Employment,Non-Public Administration Government
0,"Cheatham County, Tennessee",2010,1884.0,401.788060
1,"Davidson County, Tennessee",2010,51705.0,16277.788472


In [134]:
#the only industry is government so we are subtracting non public admin from the total for "public admin"
aside3['Public Administration'] = aside3['Government Industry Employment'] - aside3['Non-Public Administration Government']

In [135]:
aside3['Employment'] = aside3['Public Administration']
aside3['Industry'] = 'Public Administration'
aside3 = aside3[['NAME', 'Year', 'Industry', 'Employment']]

In [136]:
aside3.head(2)

,NAME,Year,Industry,Employment
0,"Cheatham County, Tennessee",2010,Public Administration,1482.21194
1,"Davidson County, Tennessee",2010,Public Administration,35427.211528


In [137]:
aside3 = aside3.pivot(index = ['NAME', 'Industry'], columns = 'Year', values = 'Employment').reset_index()
aside3.head(2)

Year,NAME,Industry,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023_2023WP,2023_2024WP,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050
0,"Cheatham County, Tennessee",Public Administration,1482.21194,1412.373521,1339.729707,1320.91488,1314.107904,1435.707328,1287.447022,1290.782223,1292.469734,1284.944707,1236.415232,1230.854729,1204.474291,1215.415368,1203.830965,1192.116816,1190.880008,1189.60393,1185.67344,1184.432125,1181.307735,1177.138609,1173.263451,1169.46357,1164.747039,1160.608406,1154.818355,1149.016023,1143.010724,1138.020762,1131.097894,1124.771028,1117.37952,1110.817168,1103.257301,1095.819326,1088.924756,1078.764159,1070.397192,1061.836201,1053.16705,1043.249853
1,"Davidson County, Tennessee",Public Administration,35427.211528,33662.572725,31475.900849,30032.743099,29386.111978,28948.097932,28571.180556,28015.222317,27058.452847,26930.15164,27883.053651,25867.865156,26262.285064,25700.588654,25922.029068,25600.937018,25029.453096,24480.724672,23923.674387,23407.576685,22858.857473,22311.869938,21776.805988,21249.187389,20732.255733,20219.181214,19714.883174,19222.774712,18738.8464,18267.304322,17803.928257,17351.81476,16910.406485,16479.849674,16057.746688,15645.292703,15240.782246,14845.121603,14456.839242,14075.68441,13700.847921,13332.759806


In [138]:
data.head(2)

,Industry,NAME,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023_2023WP,2023_2024WP,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,Share Private,Share Self-Employed,Share Government,2010 Government,2011 Government,2012 Government,2013 Government,2014 Government,2015 Government,2016 Government,2017 Government,2018 Government,2019 Government,2020 Government,2021 Government,2022 Government,2023_2023WP Government,2023_2024WP Government,2024 Government,2025 Government,2026 Government,2027 Government,2028 Government,2029 Government,2030 Government,2031 Government,2032 Government,2033 Government,2034 Government,2035 Government,2036 Government,2037 Government,2038 Government,2039 Government,2040 Government,2041 Government,2042 Government,2043 Government,2044 Government,2045 Government,2046 Government,2047 Government,2048 Government,2049 Government,2050 Government
0,Accommodation and food services,"Cheatham County, Tennessee",612.0,577.0,583.0,610.0,650.0,662.0,716.0,786.0,831.0,824.0,804.0,835.0,877.0,911.678571,910.0,942.383598,977.445767,1013.507937,1050.570106,1087.632275,1126.694444,1165.756614,1206.818783,1247.880952,1290.943122,1335.005291,1380.067460,1425.129630,1473.191799,1521.253968,1570.316138,1621.378307,1673.440476,1727.502646,1781.564815,1838.626984,1895.689153,1954.751323,2015.813492,2077.875661,2141.937831,2207.0,89.455529,2.746548,7.797922,47.723284,44.994011,45.461887,47.567326,50.686494,51.622245,55.833123,61.291669,64.800734,64.254879,62.695295,65.112651,68.387778,71.091986,70.961092,73.48634,76.220461,79.032561,81.92264,84.812719,87.858756,90.904794,94.10679,97.308786,100.666741,104.102674,107.616587,111.1305,114.878351,118.626201,122.452031,126.433819,130.493587,134.709313,138.925039,143.374702,147.824366,152.429988,157.191568,162.031128,167.026646,172.100143
1,Accommodation and food services,"Davidson County, Tennessee",41443.0,42092.0,44849.0,46526.0,49400.0,50765.0,53956.0,56158.0,59518.0,62078.0,45898.0,50924.0,59594.0,63675.392857,62294.0,62072.769841,63246.932540,64416.095238,65571.257937,66707.420635,67822.583333,68915.746032,69985.908730,71033.071429,72055.234127,73055.396825,74029.559524,74980.722222,75906.884921,76809.047619,77686.210317,78539.373016,79368.535714,80172.698413,80951.861111,81708.023810,82438.186508,83145.349206,83828.511905,84486.674603,85120.837302,85731.0,98.923177,1.076823,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [139]:
data = pd.concat([data, aside3])

In [140]:
data.head(2)

,Industry,NAME,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023_2023WP,2023_2024WP,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,Share Private,Share Self-Employed,Share Government,2010 Government,2011 Government,2012 Government,2013 Government,2014 Government,2015 Government,2016 Government,2017 Government,2018 Government,2019 Government,2020 Government,2021 Government,2022 Government,2023_2023WP Government,2023_2024WP Government,2024 Government,2025 Government,2026 Government,2027 Government,2028 Government,2029 Government,2030 Government,2031 Government,2032 Government,2033 Government,2034 Government,2035 Government,2036 Government,2037 Government,2038 Government,2039 Government,2040 Government,2041 Government,2042 Government,2043 Government,2044 Government,2045 Government,2046 Government,2047 Government,2048 Government,2049 Government,2050 Government
0,Accommodation and food services,"Cheatham County, Tennessee",612.0,577.0,583.0,610.0,650.0,662.0,716.0,786.0,831.0,824.0,804.0,835.0,877.0,911.678571,910.0,942.383598,977.445767,1013.507937,1050.570106,1087.632275,1126.694444,1165.756614,1206.818783,1247.880952,1290.943122,1335.005291,1380.06746,1425.12963,1473.191799,1521.253968,1570.316138,1621.378307,1673.440476,1727.502646,1781.564815,1838.626984,1895.689153,1954.751323,2015.813492,2077.875661,2141.937831,2207.0,89.455529,2.746548,7.797922,47.723284,44.994011,45.461887,47.567326,50.686494,51.622245,55.833123,61.291669,64.800734,64.254879,62.695295,65.112651,68.387778,71.091986,70.961092,73.48634,76.220461,79.032561,81.92264,84.812719,87.858756,90.904794,94.10679,97.308786,100.666741,104.102674,107.616587,111.1305,114.878351,118.626201,122.452031,126.433819,130.493587,134.709313,138.925039,143.374702,147.824366,152.429988,157.191568,162.031128,167.026646,172.100143
1,Accommodation and food services,"Davidson County, Tennessee",41443.0,42092.0,44849.0,46526.0,49400.0,50765.0,53956.0,56158.0,59518.0,62078.0,45898.0,50924.0,59594.0,63675.392857,62294.0,62072.769841,63246.93254,64416.095238,65571.257937,66707.420635,67822.583333,68915.746032,69985.90873,71033.071429,72055.234127,73055.396825,74029.559524,74980.722222,75906.884921,76809.047619,77686.210317,78539.373016,79368.535714,80172.698413,80951.861111,81708.02381,82438.186508,83145.349206,83828.511905,84486.674603,85120.837302,85731.0,98.923177,1.076823,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [141]:
data = data.drop(columns = ['Share Private', 'Share Government', 'Share Self-Employed'])

In [142]:
cols = ['2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', 
        '2022', '2023_2023WP', '2023_2024WP', '2024', '2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034', '2035', 
        '2036', '2037', '2038', '2039', '2040', '2041', '2042', '2043', '2044', '2045', '2046', '2047', '2048', '2049', '2050']

In [143]:
#for all columns (years) when industry is equal to public administration, the value of that equivalent year's government owned establishment employment for that 
#industry is equal to the value of public administration's value for the year (total industry employment)
#essentially all public administration employment is at a government establishment so can properly re-sum the whole so we can redistribute government emp.
for col in cols:
    data.loc[data['Industry'] == 'Public Administration', '{} Government'.format(col)] = data.loc[data['Industry'] == 'Public Administration', '{}'.format(col)]

In [144]:
data = data.melt(id_vars = ['NAME', 'Industry'], var_name = 'Year', value_name = 'Employment')

In [145]:
data.head(2)

,NAME,Industry,Year,Employment
0,"Cheatham County, Tennessee",Accommodation and food services,2010,612.0
1,"Davidson County, Tennessee",Accommodation and food services,2010,41443.0


In [146]:
#Get the total base for "Government" for each "NAME" and Year
gov_total_base = data[data['Industry'] == 'Government']
gov_total_base = gov_total_base.rename(columns = {'Employment': 'Total Government Employment'})
gov_total_base.drop(columns = 'Industry', inplace = True)

#Map this total base to the original DataFrame
data = data.merge(gov_total_base, on = ['NAME', 'Year'], how = 'left')

In [147]:
gov_total_base.head(2)

,NAME,Year,Total Government Employment
126,"Cheatham County, Tennessee",2010,1884.0
127,"Davidson County, Tennessee",2010,51705.0


In [148]:
data.head(2)

,NAME,Industry,Year,Employment,Total Government Employment
0,"Cheatham County, Tennessee",Accommodation and food services,2010,612.0,1884.0
1,"Davidson County, Tennessee",Accommodation and food services,2010,41443.0,51705.0


In [149]:
#reformat so that government establishment industry employment is a separate column
thelist = ['2010 Government', '2011 Government', '2012 Government', '2013 Government', '2014 Government', '2015 Government', 
           '2016 Government', '2017 Government', '2018 Government', '2019 Government', '2020 Government', '2021 Government',
           '2022 Government', '2023_2023WP Government', '2023_2024WP Government', '2024 Government', '2025 Government', '2026 Government', '2027 Government', 
           '2028 Government', '2029 Government', '2030 Government', '2031 Government', '2032 Government', '2033 Government', '2034 Government', 
           '2035 Government', '2036 Government', '2037 Government', '2038 Government', '2039 Government', '2040 Government', '2041 Government', 
           '2042 Government', '2043 Government', '2044 Government', '2045 Government', '2046 Government', '2047 Government', 
           '2048 Government', '2049 Government', '2050 Government']
gov = data.loc[data['Year'].isin(thelist)]
gov.drop(columns = 'Total Government Employment', inplace = True)
nongov = data.loc[~data['Year'].isin(thelist)]

In [150]:
nongov.head(2)

,NAME,Industry,Year,Employment,Total Government Employment
0,"Cheatham County, Tennessee",Accommodation and food services,2010,612.0,1884.0
1,"Davidson County, Tennessee",Accommodation and food services,2010,41443.0,51705.0


In [151]:
nongov['Year'].unique()

array(['2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017',
       '2018', '2019', '2020', '2021', '2022', '2023_2023WP',
       '2023_2024WP', '2024', '2025', '2026', '2027', '2028', '2029',
       '2030', '2031', '2032', '2033', '2034', '2035', '2036', '2037',
       '2038', '2039', '2040', '2041', '2042', '2043', '2044', '2045',
       '2046', '2047', '2048', '2049', '2050'], dtype=object)

In [152]:
gov.head(2)

,NAME,Industry,Year,Employment
14112,"Cheatham County, Tennessee",Accommodation and food services,2010 Government,47.723284
14113,"Davidson County, Tennessee",Accommodation and food services,2010 Government,0.0


In [153]:
gov['Year'].unique()

array(['2010 Government', '2011 Government', '2012 Government',
       '2013 Government', '2014 Government', '2015 Government',
       '2016 Government', '2017 Government', '2018 Government',
       '2019 Government', '2020 Government', '2021 Government',
       '2022 Government', '2023_2023WP Government',
       '2023_2024WP Government', '2024 Government', '2025 Government',
       '2026 Government', '2027 Government', '2028 Government',
       '2029 Government', '2030 Government', '2031 Government',
       '2032 Government', '2033 Government', '2034 Government',
       '2035 Government', '2036 Government', '2037 Government',
       '2038 Government', '2039 Government', '2040 Government',
       '2041 Government', '2042 Government', '2043 Government',
       '2044 Government', '2045 Government', '2046 Government',
       '2047 Government', '2048 Government', '2049 Government',
       '2050 Government'], dtype=object)

In [154]:
gov = gov.rename(columns = {'Employment': 'Industry Government Employment'})
boop = gov['Year'].str.split(pat = " ", expand = True)
gov['Year'] = boop[0].str.strip()

In [155]:
data = nongov.merge(gov, on = ['NAME', 'Industry', 'Year'])

In [156]:
data.head(2)

,NAME,Industry,Year,Employment,Total Government Employment,Industry Government Employment
0,"Cheatham County, Tennessee",Accommodation and food services,2010,612.0,1884.0,47.723284
1,"Davidson County, Tennessee",Accommodation and food services,2010,41443.0,51705.0,0.0


In [157]:
data['Year'].unique()

array(['2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017',
       '2018', '2019', '2020', '2021', '2022', '2023_2023WP',
       '2023_2024WP', '2024', '2025', '2026', '2027', '2028', '2029',
       '2030', '2031', '2032', '2033', '2034', '2035', '2036', '2037',
       '2038', '2039', '2040', '2041', '2042', '2043', '2044', '2045',
       '2046', '2047', '2048', '2049', '2050'], dtype=object)

In [158]:
#Calculate the "Share of Total Government Employment"
#This is done by dividing the 'Government' by the 'Total Gov Base' for each row
data['Share of Total Government Employment'] = (data['Industry Government Employment'] / data['Total Government Employment'])*100

In [159]:
data.head(2)

,NAME,Industry,Year,Employment,Total Government Employment,Industry Government Employment,Share of Total Government Employment
0,"Cheatham County, Tennessee",Accommodation and food services,2010,612.0,1884.0,47.723284,2.533083
1,"Davidson County, Tennessee",Accommodation and food services,2010,41443.0,51705.0,0.0,0.0


In [160]:
#hardcode the value of share of total government employment to be 0 for both total and government employment
data.loc[data['Industry'] == 'Total employment (number of jobs)', 'Share of Total Government Employment'] = 0
data.loc[data['Industry'] == 'Government', 'Share of Total Government Employment'] = 0

In [161]:
#multiply value of total government employment by the shares
data.loc[data['Industry'] == 'Public Administration', 'Employment'] = 0
data['Extra Government Employment'] = (data['Share of Total Government Employment']/100) * data['Total Government Employment']
data['NAICS Employment'] = data['Employment'] + data['Extra Government Employment']
#hardcode government NAICS employment to be 0 as an extra measure
data.loc[data['Industry'] == 'Government', 'NAICS Employment'] = 0

In [162]:
catdict = {'Agriculture, forestry, fishing and hunting': 'BEA and NAICS', 'Government': 'BEA', 'Total employment (number of jobs)': 'BEA and NAICS', 
           'Mining, quarrying, and oil and gas extraction': 'BEA and NAICS', 'Utilities': 'BEA and NAICS', 'Construction': 'BEA and NAICS', 
           'Manufacturing': 'BEA and NAICS', 'Wholesale trade': 'BEA and NAICS', 'Retail trade': 'BEA and NAICS', 
           'Transportation and warehousing': 'BEA and NAICS', 'Information': 'BEA and NAICS', 'Finance and insurance': 'BEA and NAICS', 
           'Real estate and rental and leasing': 'BEA and NAICS', 'Professional, scientific, and technical services': 'BEA and NAICS', 
           'Management of companies and enterprises': 'BEA and NAICS', 
           'Administrative and support and waste management and remediation services': 'BEA and NAICS', 'Educational services': 'BEA and NAICS', 
           'Health care and social assistance': 'BEA and NAICS', 'Arts, entertainment, and recreation': 'BEA and NAICS', 
           'Accommodation and food services': 'BEA and NAICS', 'Other services (except government and government enterprises)': 'BEA and NAICS', 
           'Public Administration': 'NAICS', 'Farm employment': 'BEA and NAICS', 'Forestry, fishing, and related activities': 'BEA and NAICS'}
naicsdict = {'Agriculture, forestry, fishing and hunting': '11', 'Government': 'NA', 'Total employment (number of jobs)': 'NA', 
           'Mining, quarrying, and oil and gas extraction': '21', 'Utilities': '22', 'Construction': '23', 'Manufacturing': '31-33', 
           'Wholesale trade': '42', 'Retail trade': '44-45', 'Transportation and warehousing': '48-49', 'Information': '51', 
           'Finance and insurance': '52', 'Real estate and rental and leasing': '53', 'Professional, scientific, and technical services': '54', 
           'Management of companies and enterprises': '55', 'Administrative and support and waste management and remediation services': '56', 
           'Educational services': '61', 'Health care and social assistance': '62', 'Arts, entertainment, and recreation': '71', 
           'Accommodation and food services': '72', 'Other services (except government and government enterprises)': '81', 'Public Administration': '92', 
             'Farm employment': '111, 112', 'Forestry, fishing, and related activities': '113, 114, 115'}
beadict = {'Agriculture, forestry, fishing and hunting': '70, 100', 'Government': '2000', 'Total employment (number of jobs)': '10', 
           'Mining, quarrying, and oil and gas extraction': '200', 'Utilities': '300', 'Construction': '400', 'Manufacturing': '500', 
           'Wholesale trade': '600', 'Retail trade': '700', 'Transportation and warehousing': '800', 'Information': '900', 'Finance and insurance': '1000', 
           'Real estate and rental and leasing': '1100', 'Professional, scientific, and technical services': '1200', 
           'Management of companies and enterprises': '1300', 'Administrative and support and waste management and remediation services': '1400', 
           'Educational services': '1500', 'Health care and social assistance': '1600', 'Arts, entertainment, and recreation': '1700', 
           'Accommodation and food services': '1800', 'Other services (except government and government enterprises)': '1900', 'Public Administration': 'NA', 
          'Farm employment': '70', 'Forestry, fishing, and related activities': '100'}

In [163]:
data['NAICS Code'] = data['Industry'].map(naicsdict)
data['BEA Line Code'] = data['Industry'].map(beadict)
data['Category'] = data['Industry'].map(catdict)

In [164]:
data.head(2)

,NAME,Industry,Year,Employment,Total Government Employment,Industry Government Employment,Share of Total Government Employment,Extra Government Employment,NAICS Employment,NAICS Code,BEA Line Code,Category
0,"Cheatham County, Tennessee",Accommodation and food services,2010,612.0,1884.0,47.723284,2.533083,47.723284,659.723284,72,1800,BEA and NAICS
1,"Davidson County, Tennessee",Accommodation and food services,2010,41443.0,51705.0,0.0,0.0,0.0,41443.0,72,1800,BEA and NAICS


In [165]:
data['Year'].unique()

array(['2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017',
       '2018', '2019', '2020', '2021', '2022', '2023_2023WP',
       '2023_2024WP', '2024', '2025', '2026', '2027', '2028', '2029',
       '2030', '2031', '2032', '2033', '2034', '2035', '2036', '2037',
       '2038', '2039', '2040', '2041', '2042', '2043', '2044', '2045',
       '2046', '2047', '2048', '2049', '2050'], dtype=object)

In [166]:
data = data.rename(columns = {'Employment': 'BEA Employment'}).sort_values(by = 'NAME')

In [167]:
data = data[['NAME', 'Year', 'Industry', 'Category', 'NAICS Code', 'BEA Line Code', 'BEA Employment', 'NAICS Employment']]

In [168]:
#data.to_csv('../../Comprehensive-Plans/AdjustedYoYEmpl_BLandNAICS_2024WP.csv', index = False)

# Mantissa Round

In [169]:
df = data.pivot(columns = 'NAME', index = ['Industry', 'Year'], values = 'NAICS Employment')
df.reset_index(drop = False, inplace = True)
df = df.loc[df['Industry'] != 'Government']

In [170]:
df.head(2)

NAME,Industry,Year,"Cheatham County, Tennessee","Davidson County, Tennessee","Dickson County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Maury County, Tennessee","Montgomery County, Tennessee","Robertson County, Tennessee","Rutherford County, Tennessee","Stewart County, Tennessee","Sumner County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Wilson County, Tennessee"
0,Accommodation and food services,2010,659.723284,41443.0,1351.0,127.0,445.0,2869.881455,6590.0,1489.0,9881.0,229.0,3991.0,126.0,9383.0,4034.0
1,Accommodation and food services,2011,621.994011,42092.0,1433.0,133.0,433.0,2886.193393,6985.0,1485.0,10387.0,227.0,4230.0,132.0,10014.0,3773.0


In [171]:
years = list(df['Year'].unique())

In [172]:
cols = list(df.columns)
cols.remove('Industry')
cols.remove('Year')
df[cols] = df[cols].astype(float)
df = df.fillna(0)

In [173]:
#separate the total as this shouldn't be included in the mantissa
totes = df.loc[df['Industry'] == 'Total employment (number of jobs)']
df = df.loc[df['Industry'] != 'Total employment (number of jobs)']

In [174]:
df.head()

NAME,Industry,Year,"Cheatham County, Tennessee","Davidson County, Tennessee","Dickson County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Maury County, Tennessee","Montgomery County, Tennessee","Robertson County, Tennessee","Rutherford County, Tennessee","Stewart County, Tennessee","Sumner County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Wilson County, Tennessee"
0,Accommodation and food services,2010,659.723284,41443.0,1351.0,127.0,445.0,2869.881455,6590.0,1489.0,9881.0,229.0,3991.0,126.0,9383.0,4034.0
1,Accommodation and food services,2011,621.994011,42092.0,1433.0,133.0,433.0,2886.193393,6985.0,1485.0,10387.0,227.0,4230.0,132.0,10014.0,3773.0
2,Accommodation and food services,2012,628.461887,44849.0,1365.0,127.0,439.0,2946.343661,7386.0,1545.0,10781.0,257.0,4641.0,141.0,11284.0,4159.0
3,Accommodation and food services,2013,657.567326,46526.0,1452.0,124.0,488.0,3205.295665,7669.0,1640.0,11570.0,224.0,4824.0,137.0,11191.0,4449.0
4,Accommodation and food services,2014,700.686494,49400.0,1481.0,109.0,480.0,3211.412641,8073.0,1736.0,12000.0,232.0,4991.0,162.0,11534.0,4938.0


In [175]:
# Create an empty list to collect dataframes
df_list = []
for year in years:
    df_filtered = df.loc[df['Year'] == year]
    df_filtered.reset_index(drop=False, inplace=True)
    cols = ['Cheatham County, Tennessee', 'Davidson County, Tennessee',
            'Dickson County, Tennessee', 'Houston County, Tennessee',
            'Humphreys County, Tennessee', 'Maury County, Tennessee',
            'Montgomery County, Tennessee', 'Robertson County, Tennessee',
            'Rutherford County, Tennessee', 'Stewart County, Tennessee',
            'Sumner County, Tennessee', 'Trousdale County, Tennessee',
            'Williamson County, Tennessee', 'Wilson County, Tennessee']
    for col in cols:
        df_filtered['{}'.format(col)] = mantissa_round(df_filtered['{}'.format(col)])
    # Append the processed dataframe to the list
    df_list.append(df_filtered)
# Concatenate all dataframes in the list
final_df = pd.concat(df_list, ignore_index=True)

In [176]:
final_df.head(2)

NAME,index,Industry,Year,"Cheatham County, Tennessee","Davidson County, Tennessee","Dickson County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Maury County, Tennessee","Montgomery County, Tennessee","Robertson County, Tennessee","Rutherford County, Tennessee","Stewart County, Tennessee","Sumner County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Wilson County, Tennessee"
0,0,Accommodation and food services,2010,659.0,41444.0,1352.0,128.0,446.0,2869.0,6591.0,1490.0,9882.0,230.0,3992.0,127.0,9384.0,4035.0
1,42,Administrative and support and waste managemen...,2010,1200.0,36944.0,1143.0,85.0,232.0,2120.0,3857.0,2068.0,9897.0,177.0,3853.0,121.0,8896.0,3117.0


In [177]:
totes.head(2)

NAME,Industry,Year,"Cheatham County, Tennessee","Davidson County, Tennessee","Dickson County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Maury County, Tennessee","Montgomery County, Tennessee","Robertson County, Tennessee","Rutherford County, Tennessee","Stewart County, Tennessee","Sumner County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Wilson County, Tennessee"
840,Total employment (number of jobs),2010,14015.0,516004.0,21408.0,2892.0,8377.0,41315.0,63879.0,27326.0,132284.0,4720.0,65038.0,2553.0,136018.0,53289.0
841,Total employment (number of jobs),2011,13764.0,527727.0,22345.0,2906.0,8377.0,41693.0,65643.0,27980.0,136703.0,4726.0,67358.0,2541.0,142959.0,53854.0


In [178]:
final_df.drop(columns = 'index', inplace = True)
final_df = pd.concat([final_df, totes])

In [179]:
final_df.head()

NAME,Industry,Year,"Cheatham County, Tennessee","Davidson County, Tennessee","Dickson County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Maury County, Tennessee","Montgomery County, Tennessee","Robertson County, Tennessee","Rutherford County, Tennessee","Stewart County, Tennessee","Sumner County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Wilson County, Tennessee"
0,Accommodation and food services,2010,659.0,41444.0,1352.0,128.0,446.0,2869.0,6591.0,1490.0,9882.0,230.0,3992.0,127.0,9384.0,4035.0
1,Administrative and support and waste managemen...,2010,1200.0,36944.0,1143.0,85.0,232.0,2120.0,3857.0,2068.0,9897.0,177.0,3853.0,121.0,8896.0,3117.0
2,"Agriculture, forestry, fishing and hunting",2010,514.0,680.0,1240.0,419.0,677.0,1644.0,985.0,1749.0,1575.0,426.0,1730.0,303.0,1555.0,1612.0
3,"Arts, entertainment, and recreation",2010,545.0,19786.0,337.0,11.0,83.0,815.0,779.0,494.0,2232.0,51.0,1900.0,13.0,5897.0,1204.0
4,Construction,2010,1501.0,24031.0,2038.0,309.0,650.0,2472.0,4074.0,2271.0,7365.0,658.0,4859.0,152.0,6665.0,3943.0


In [180]:
data = final_df

In [181]:
GNRCCounties = [data[('Stewart County, Tennessee')],data[('Montgomery County, Tennessee')],
                data[('Houston County, Tennessee')],data[('Humphreys County, Tennessee')],
                data[('Dickson County, Tennessee')],data[('Cheatham County, Tennessee')],
                data[('Robertson County, Tennessee')],data[('Sumner County, Tennessee')],
                data[('Davidson County, Tennessee')],data[('Wilson County, Tennessee')],
                data[('Trousdale County, Tennessee')],data[('Williamson County, Tennessee')],
                data[('Rutherford County, Tennessee')]]
data['GNRC'] = sum(GNRCCounties)
GNRCCountiesAll = [data[('Stewart County, Tennessee')],data[('Montgomery County, Tennessee')],
                data[('Houston County, Tennessee')],data[('Humphreys County, Tennessee')],
                data[('Dickson County, Tennessee')],data[('Cheatham County, Tennessee')],
                data[('Robertson County, Tennessee')],data[('Sumner County, Tennessee')],
                data[('Davidson County, Tennessee')],data[('Wilson County, Tennessee')],
                data[('Trousdale County, Tennessee')],data[('Williamson County, Tennessee')],
                data[('Rutherford County, Tennessee')], data['Maury County, Tennessee']]
data['GNRC Region'] = sum(GNRCCountiesAll)
MPOCounties = [data[('Robertson County, Tennessee')],data[('Sumner County, Tennessee')],
               data[('Davidson County, Tennessee')],data[('Wilson County, Tennessee')],
               data[('Williamson County, Tennessee')],data[('Rutherford County, Tennessee')],
               data[('Maury County, Tennessee')]]
data['MPO'] = sum(MPOCounties)

In [182]:
#final_df.to_csv('../data/TwoDigit_FinalCountyLevel_2024WP.csv', index = False)

In [183]:
newagg = data.melt(id_vars = ['Industry', 'Year'], var_name = 'NAME', value_name = 'Employment')
newagg = newagg.pivot(index = ['Year', 'NAME'], columns = 'Industry', values = 'Employment').reset_index(drop = False)

thelist = [newagg['Utilities'], newagg['Manufacturing'], newagg['Wholesale trade'], newagg['Transportation and warehousing']]
newagg['Industrial'] = sum(thelist)
newagg = newagg.drop(columns = ['Utilities', 'Manufacturing', 'Wholesale trade', 'Transportation and warehousing'])
thelist = [newagg['Information'], newagg['Finance and insurance'], newagg['Real estate and rental and leasing'], 
           newagg['Professional, scientific, and technical services'], newagg['Management of companies and enterprises'], 
           newagg['Administrative and support and waste management and remediation services']]
newagg['Office'] = sum(thelist)
newagg = newagg.drop(columns = ['Information', 'Finance and insurance', 'Real estate and rental and leasing', 'Professional, scientific, and technical services', 
                                'Management of companies and enterprises', 'Administrative and support and waste management and remediation services'])
thelist = [newagg['Arts, entertainment, and recreation'], newagg['Other services (except government and government enterprises)']]
newagg['Service'] = sum(thelist)
newagg = newagg.drop(columns = ['Arts, entertainment, and recreation', 'Other services (except government and government enterprises)'])
thelist = [newagg['Agriculture, forestry, fishing and hunting'], newagg['Mining, quarrying, and oil and gas extraction'], newagg['Construction']]
newagg['Other'] = sum(thelist)
newagg = newagg.drop(columns = ['Agriculture, forestry, fishing and hunting', 'Mining, quarrying, and oil and gas extraction', 'Construction'])
newagg['Education'] = newagg['Educational services']
newagg['Food Services'] = newagg['Accommodation and food services']
newagg['Government'] = newagg['Public Administration']
newagg['Medical'] = newagg['Health care and social assistance']
newagg['Retail'] = newagg['Retail trade']
newagg = newagg.drop(columns = ['Educational services', 'Accommodation and food services', 'Public Administration', 'Health care and social assistance', 
                                'Retail trade', 'Farm employment', 'Forestry, fishing, and related activities'])

In [184]:
newagg.head()

Industry,Year,NAME,Total employment (number of jobs),Industrial,Office,Service,Other,Education,Food Services,Government,Medical,Retail
0,2010,"Cheatham County, Tennessee",14015.0,3251.0,2866.0,1516.0,2031.0,326.0,659.0,1482.0,636.0,1249.0
1,2010,"Davidson County, Tennessee",516004.0,68027.0,141086.0,48158.0,25582.0,33311.0,41444.0,35427.0,74733.0,48235.0
2,2010,"Dickson County, Tennessee",21408.0,3600.0,3638.0,1660.0,3351.0,320.0,1352.0,2411.0,2350.0,2726.0
3,2010,GNRC,1047803.0,151563.0,278156.0,94809.0,74030.0,46330.0,79760.0,88482.0,122862.0,111813.0
4,2010,GNRC Region,1089118.0,156915.0,287395.0,98517.0,78236.0,47051.0,82629.0,93275.0,128431.0,116671.0


In [185]:
newagg.to_csv('../data/WP23V_IndEmp_Custom22_vs_WP24V_IndEmp_WP22.csv', index = False)

In [186]:
newagg['Year'].unique()

array(['2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017',
       '2018', '2019', '2020', '2021', '2022', '2023_2023WP',
       '2023_2024WP', '2024', '2025', '2026', '2027', '2028', '2029',
       '2030', '2031', '2032', '2033', '2034', '2035', '2036', '2037',
       '2038', '2039', '2040', '2041', '2042', '2043', '2044', '2045',
       '2046', '2047', '2048', '2049', '2050'], dtype=object)

In [187]:
newagg = newagg.loc[newagg['Year'] != '2023_2023WP']
newagg['Year'] = newagg['Year'].replace({'2023_2024WP': '2023'})

In [188]:
newagg['Year'] = newagg['Year'].astype(float)

In [189]:
data = newagg.melt(id_vars = ['NAME', 'Year'], var_name = 'Industry', value_name = 'Employment')
data = data.loc[data['Year'] > 2022]
data['WP Vintage'] = '2024'

In [190]:
data.head()

,NAME,Year,Industry,Employment,WP Vintage
221,"Cheatham County, Tennessee",2023.0,Total employment (number of jobs),1.762896e+04,2024
222,"Davidson County, Tennessee",2023.0,Total employment (number of jobs),7.547160e+05,2024
223,"Dickson County, Tennessee",2023.0,Total employment (number of jobs),2.895400e+04,2024
224,GNRC,2023.0,Total employment (number of jobs),1.619159e+06,2024
225,GNRC Region,2023.0,Total employment (number of jobs),1.679723e+06,2024


In [192]:
data.to_csv('../data/EMPLOYMENT_WP24V.csv', index = False)